In [2]:
import logging
from google.cloud import bigquery

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

PROJECT_ID = "tz-data-dev"
LOCATION = "europe-west2"
CLIENT = bigquery.Client()

def load_csv_to_bigquery_with_archive_link(
    dataset_id: str,
    table_name: str,
    gcs_uri: str,
    archive_link: str,
    skip_leading_rows: int,
    write_disposition: str,
    partition_field: str = None,
    enable_character_map_v2: bool = False,
    client: bigquery.Client = CLIENT,
    project_id: str = PROJECT_ID,
) -> None:
    """
    Helper function to load CSV data into BigQuery with an archive_link column,
    optionally supporting partitioned tables.
    Parameters:
        client (bigquery.Client): BigQuery client instance.
        project_id (str): The GCP project ID.
        dataset_id (str): The BigQuery dataset ID.
        table_name (str): The name of the BigQuery table.
        gcs_uri (str): The GCS URI of the CSV file to load.
        archive_link (str): The archive link to add to the BigQuery table.
        skip_leading_rows (int): Number of header rows to skip in the CSV file.
        write_disposition (str): BigQuery write disposition, e.g., WRITE_TRUNCATE or WRITE_APPEND.
        partition_field (str, optional): Column name to use for partitioning the table.
        enable_character_map_v2 (bool, optional): Flag to enable character map V2. Defaults to False.
    Returns:
        None
    """
    table_id = f"{project_id}.{dataset_id}.{table_name}"
    # Configure job with schema auto-detection enabled and parameterized options
    job_config = bigquery.LoadJobConfig(
        autodetect=True,
        source_format=bigquery.SourceFormat.CSV,
        skip_leading_rows=skip_leading_rows,
        write_disposition=write_disposition,
    )
    # Apply partitioning configuration if partition_field is specified
    if partition_field:
        job_config.time_partitioning = bigquery.TimePartitioning(
            type_=bigquery.TimePartitioningType.DAY, field=partition_field
        )
        logger.info(f"Using partition field: {partition_field}")
    # Enable Character Map V2 if specified
    if enable_character_map_v2:
        job_config.column_name_character_map = "V2"  # Enable column name character map V2
        logger.info("Character Map V2 is enabled.")
    # Load data from GCS to BigQuery table
    load_job = client.load_table_from_uri(gcs_uri, table_id, job_config=job_config)
    load_job.result()  # Wait for the job to complete
    logger.info(f"Loaded data from {gcs_uri} to BigQuery table {table_id}")
    # Add the archive_link column to the table
    add_archive_link_column = f"""
    ALTER TABLE {table_id}
    ADD COLUMN IF NOT EXISTS archive_link STRING
    """
    add_archive_column_data = f"""
    UPDATE {table_id}
    SET archive_link = '{archive_link}'
    WHERE TRUE
    """
    client.query(add_archive_link_column).result()
    client.query(add_archive_column_data).result()
    logger.info(f"Added archive_link column to table {table_id} with value {archive_link}")

In [6]:
def load_csv_to_bigquery(
    dataset_id: str,
    table_name: str,
    file_path: str,
    skip_leading_rows: int,
    write_disposition: str,
    client: bigquery.Client = CLIENT,
    project_id: str = PROJECT_ID,

) -> None:
    """
    Loads a local CSV file into a BigQuery table and adds an archive_link column.
    Parameters:
        dataset_id (str): The BigQuery dataset ID.
        table_name (str): The target BigQuery table name.
        file_path (str): Local path to the CSV file.
        skip_leading_rows (int): Number of header rows to skip.
        write_disposition (str): BigQuery write disposition (e.g., WRITE_APPEND, WRITE_TRUNCATE).
        client (bigquery.Client): BigQuery client instance.
        project_id (str): Google Cloud project ID.
    Returns:
        None
    """
    table_id = f"{project_id}.{dataset_id}.{table_name}"
    job_config = bigquery.LoadJobConfig(
        autodetect=True,
        source_format=bigquery.SourceFormat.CSV,
        skip_leading_rows=skip_leading_rows,
        write_disposition=write_disposition,
    )
    with open(file_path, "rb") as file_obj:
        load_job = client.load_table_from_file(file_obj, table_id, job_config=job_config)
        load_job.result()  # Wait for job to complete
    logger.info(f"Loaded data from local file {file_path} to BigQuery table {table_id}")

In [8]:
from google.api_core.exceptions import Conflict, NotFound

In [11]:
load_csv_to_bigquery(
    dataset_id="test_dataset",
    table_name="Taiwan_sample_vis_output",
    file_path="C:/Users/jy/OneDrive - TransitionZero/tza-pypsa/twn_sample_vis_output.csv",
    skip_leading_rows=1,
    write_disposition="WRITE_APPEND",
)

Forbidden: 403 POST https://bigquery.googleapis.com/upload/bigquery/v2/projects/tz-data-dev/jobs?uploadType=resumable: Access Denied: Dataset tz-data-dev:test_dataset: Permission bigquery.tables.create denied on dataset tz-data-dev:test_dataset (or it may not exist).

In [9]:
def dataset_exists(dataset_id: str, project_id: str = PROJECT_ID) -> bool:
    """
    Checks whether a dataset exists in the specified BigQuery project.
    Args:
        dataset_id (str): The ID of the dataset to check.
        project_id (str, optional): The ID of the Google Cloud project. Defaults to the global PROJECT_ID.
    Returns:
        bool: True if the dataset exists, False otherwise.
    """
    dataset_ref = f"{project_id}.{dataset_id}"
    try:
        CLIENT.get_dataset(dataset_ref)
        logger.info(f"Dataset {dataset_id} already exists.")
        return True
    except NotFound:
        logger.info(f"Dataset {dataset_id} does not exist. It will be created.")
        return False

In [10]:
dataset_exists("test_dataset", PROJECT_ID)

INFO:__main__:Dataset test_dataset does not exist. It will be created.


False